# Malinois motif coverage: which motifs still need a CAV *(CPU runtime)*

Question: if we want a function CAV for **every motif that occurs in the Malinois variants**, how many are already
computed and how many are left?

1. Scan every Malinois variant with **every JASPAR matrix** (FIMO p < `fimo_p`, both strands, hits that contain the
   variant), as `Malinois MPRA 6` 2.1 did for the new TFs.
2. Give each matrix a status:
   - **done**: one of the 251 TFs with sites and fits in `fcav_pool` (Malinois P(bound) exists);
   - **sites built, not fitted / fitted, no Malinois P**: new TFs from `Malinois MPRA 6` (`fcav_more`), by how far they got;
   - **tried, too few sites**: a sites file exists but has fewer than 50 matched pairs;
   - **has ChIP, not started**: ENCODE has TF ChIP-seq for this name in HepG2 or K562, nothing built yet;
   - **no ChIP in HepG2/K562**: cannot get a CAV with the current recipe;
   - **dimer matrix** (`A::B`): skipped by the recipe.
3. Count matrices, occurrences and variants per status, and how many variants (all / active / clear effect, per
   cell) are covered now vs if every computable motif were done.

Nothing is fitted or written except the scan table (`malinois/all_motif_hits.csv.gz`) and the status table
(`malinois/motif_status.csv`). The scan is resumable in blocks of matrices.

In [ ]:
!pip install -q pymemesuite pyfaidx
from google.colab import drive
drive.mount('/content/drive')

## 1. Setup: matrices, genome, Malinois sequences, ENCODE ChIP targets

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil, requests
import numpy as np, pandas as pd
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO
from pyfaidx import Fasta

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL, MORE, MAL = f"{WORK_DIR}/fcav_pool", f"{WORK_DIR}/fcav_more", f"{WORK_DIR}/malinois"
SCAN = f"{MAL}/all_motif_scan"                    # per-block scan results (resumable)
os.makedirs(SCAN, exist_ok=True)
VPOS, MOFF, WLEN = 99, C["motif_offset"], C["window_len"]
CELLS = ["HepG2", "K562"]
pd.set_option("display.width", 250)

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
WID = {m: np.asarray(MOTIF[m].frequencies).shape[0] for m in MOTIF}
print(f"{len(MOTIF)} JASPAR matrices ({sum('::' in m for m in MOTIF)} dimers)")

LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
CHROM_LEN = {c: len(GENOME[c]) for c in GENOME.keys()}

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
assert (V.index.values == np.arange(len(V))).all()
SEQV = [None] * len(V)
for ch, g in V.groupby("chrom38"):
    s = GENOME[ch][:].seq
    for r_, p in zip(g.index.values, g["pos38"].values):
        SEQV[r_] = s[p - VPOS:p - VPOS + 200]
    del s
assert all(x[VPOS] == b for x, b in zip(SEQV, V["ref"])), "hg38 base at the variant differs from the reference allele"
print(f"{len(V):,} Malinois variants")

# ENCODE TF ChIP-seq targets per cell (copied from Pooled Regions 1)
ENC, HDR = "https://www.encodeproject.org", {"accept": "application/json"}


def enc_search(**params):
    r = requests.get(f"{ENC}/search/", params={**params, "format": "json", "limit": "all"}, headers=HDR, timeout=600)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return r.json()["@graph"]


def cell_targets(cell):
    exps = enc_search(type="Experiment", assay_title="TF ChIP-seq", status="released",
                      **{"biosample_ontology.term_name": cell}, field="target.label")
    return {e["target"]["label"].upper(): e["target"]["label"] for e in exps if "target" in e}


LABEL = {c: cell_targets(c) for c in CELLS}
print("  ".join(f"{c}: {len(LABEL[c])} ChIP-seq targets" for c in CELLS))

## 2. Scan every matrix (resumable)

Only the bases a motif containing base 100 can reach are scanned (as in the Malinois notebooks). Blocks of
`BLOCK` matrices are saved as they finish; a restarted session skips finished blocks.

In [ ]:
BLOCK = 40
NAMES = sorted(MOTIF)
MAXW = max(WID.values())
LO_, HI_ = VPOS - MAXW + 1, VPOS + MAXW
CHUNKS = []
_sub = [s[LO_:HI_] for s in SEQV]
for i0 in range(0, len(_sub), 5000):
    CHUNKS.append([Sequence(x, name=str(i0 + k).encode()) for k, x in enumerate(_sub[i0:i0 + 5000])])
del _sub
t0 = time.time()
blocks = [NAMES[i:i + BLOCK] for i in range(0, len(NAMES), BLOCK)]
for bi, names in enumerate(blocks):
    out = f"{SCAN}/block_{bi:03d}.csv.gz"
    if os.path.exists(out):
        continue
    rows = []
    for m in names:
        for ch in CHUNKS:
            for h in FIMO_SCAN.score_motif(MOTIF[m], ch, BACKGROUND).matched_elements:
                a, b = sorted((h.start, h.stop))
                a, b = a - 1 + LO_, b + LO_
                if a <= VPOS < b:
                    rows.append((int(h.source.name.decode()), m, a, b, h.strand, h.pvalue))
    pd.DataFrame(rows, columns=["vi", "motif", "a", "b", "strand", "p"]).to_csv(out + ".part", index=False, compression="gzip")
    os.replace(out + ".part", out)
    el = time.time() - t0
    print(f"block {bi + 1}/{len(blocks)}: {len(rows):,} hits  [{el / 60:.1f} min]", flush=True)
HA = pd.concat([pd.read_csv(f"{SCAN}/block_{bi:03d}.csv.gz") for bi in range(len(blocks))], ignore_index=True)
HA["chrom38"] = V["chrom38"].values[HA["vi"]]
HA["g38"] = V["pos38"].values[HA["vi"]] - VPOS + HA["a"]
HA["occ"] = HA["chrom38"] + ":" + HA["g38"].astype(str) + ":" + HA["strand"] + ":" + HA["motif"]
HA.to_csv(f"{MAL}/all_motif_hits.csv.gz", index=False, compression="gzip")
print(f"{len(HA):,} (variant, hit) pairs; {HA['vi'].nunique():,} variants in at least one hit; "
      f"{HA['occ'].nunique():,} occurrences; {HA['motif'].nunique()} matrices with a hit")

## 3. Status of every matrix with a hit

In [ ]:
def sites_status(d, m):
    f = f"{d}/sites/{m}.json"
    if not os.path.exists(f):
        return None
    return json.load(open(f))["status"]


def n_files(d, ext):
    return {f[:-len(ext)] for f in os.listdir(d) if f.endswith(ext) and not f.endswith(".part" + ext)} if os.path.isdir(d) else set()


FIT_POOL = {t: n_files(f"{POOL}/fits/{t}", ".npz") for t in ("nt500m", "hyenadna", "dnabert2", "genalm", "nt2p5b")}
FIT_SWEEP = {t: n_files(f"{POOL}/fits_layers/all/{t}", ".npz") for t in ("nt500m", "hyenadna", "dnabert2", "genalm")}
FIT_MORE = n_files(f"{MORE}/fits/nt500m", ".npz")
P_OLD = n_files(f"{MAL}/p_nt500m", ".csv")
P_MORE = n_files(f"{MAL}/more/p_ref", ".csv")


def status(m):
    if "::" in m:
        return "dimer matrix"
    s = sites_status(POOL, m)
    if s == "built":
        return "done (251 TFs)" if m in P_OLD or m in FIT_POOL["nt500m"] else "sites built (pool), not fitted"
    s2 = sites_status(MORE, m)
    if s2 == "built":
        if m in P_MORE:
            return "new TF: fitted, Malinois P done"
        return "new TF: fitted, no Malinois P" if m in FIT_MORE else "new TF: sites built, not fitted"
    if (s and s != "built") or (s2 and s2 != "built"):
        return "tried, too few sites"
    if any(m.upper() in LABEL[c] for c in CELLS):
        return "has ChIP, not started"
    return "no ChIP in HepG2/K562"


g = HA.groupby("motif")
ST = pd.DataFrame({"pairs": g.size(), "variants": g["vi"].nunique(), "occurrences": g["occ"].nunique()})
ST["status"] = [status(m) for m in ST.index]
ST["matrix"] = ["human (all caps)" if m.upper() == m else "mouse/other name" for m in ST.index]
ST["ChIP cells"] = ["+".join(c for c in CELLS if m.upper() in LABEL[c]) for m in ST.index]
for t, s in FIT_SWEEP.items():
    ST[f"layer fits {t}"] = ST.index.isin(s)
ST.to_csv(f"{MAL}/motif_status.csv")
ORDER_ST = ["done (251 TFs)", "new TF: fitted, Malinois P done", "new TF: fitted, no Malinois P", "new TF: sites built, not fitted",
            "sites built (pool), not fitted", "has ChIP, not started", "tried, too few sites", "no ChIP in HepG2/K562", "dimer matrix"]
S = ST.groupby("status").agg(matrices=("pairs", "size"), occurrences=("occurrences", "sum"), pairs=("pairs", "sum"))
S = S.reindex([o for o in ORDER_ST if o in S.index])
S["share of occurrences"] = S["occurrences"] / S["occurrences"].sum()
print(f"{len(ST)} matrices with at least one hit containing a Malinois variant\n")
print(S.to_string(float_format=lambda x: f"{x:.2f}"))
print("\nhuman vs other matrix names, per status:\n" + pd.crosstab(ST["status"], ST["matrix"]).reindex(S.index).to_string())
todo = ST[ST["status"] == "has ChIP, not started"].sort_values("occurrences", ascending=False)
print(f"\nhas ChIP, not started: {len(todo)} matrices, {todo['occurrences'].sum():,} occurrences; the 30 with most occurrences:")
print(todo.head(30)[["occurrences", "variants", "ChIP cells", "matrix"]].to_string())

## 4. How many variants are covered

A variant is covered by a group of matrices if at least one hit of those matrices contains it. Shown for all
variants, active ones (log2FC ≥ 1 for one allele) and clear effects (|Δ| ≥ 2 SE, active), per cell:
- **now**: the 251 TFs;
- **+ new TFs already built** (`fcav_more`, any stage);
- **+ everything with ChIP** (adds "has ChIP, not started" and "sites built (pool), not fitted");
- **any matrix**: the ceiling of a motif-based method with this motif set.

In [ ]:
GROUPS = {
    "now (251 TFs)": ["done (251 TFs)"],
    "+ new TFs already built": ["done (251 TFs)", "new TF: fitted, Malinois P done", "new TF: fitted, no Malinois P",
                                "new TF: sites built, not fitted"],
    "+ everything with ChIP": ["done (251 TFs)", "new TF: fitted, Malinois P done", "new TF: fitted, no Malinois P",
                               "new TF: sites built, not fitted", "sites built (pool), not fitted", "has ChIP, not started"],
    "any matrix": ORDER_ST,
}
rows = []
for gname, sts in GROUPS.items():
    ms = set(ST.index[ST["status"].isin(sts)])
    cov = np.zeros(len(V), bool)
    cov[HA.loc[HA["motif"].isin(ms), "vi"].unique()] = True
    r = {"group": gname, "matrices": len(ms), "variants": int(cov.sum()), "share of all": cov.mean()}
    for c in ("HepG2", "K562", "SKNSH"):
        act = V[f"act_{c}"].values >= 1.0
        clear = act & (np.abs(V[f"d_{c}"].values) / V[f"se_{c}"].values >= 2.0)
        r[f"{c} active"] = cov[act].mean()
        r[f"{c} clear"] = cov[clear].mean()
    rows.append(r)
print(pd.DataFrame(rows).set_index("group").to_string(float_format=lambda x: f"{x:.3f}"))

## End

`malinois/all_motif_hits.csv.gz` (every JASPAR hit containing a Malinois variant) and `malinois/motif_status.csv`
(one row per matrix: hits, occurrences, status, ChIP cells, whether layer-sweep fits exist).